In [ ]:
import tifffile
import cv2
import os
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
from pathlib import Path
import shutil


In [ ]:
prepared_dir = Path("PATH/TO/PREPARED_INPUT")
superst_dir = Path("PATH/TO/SUPERST_INPUT")
pixel_size = float((prepared_dir / "pixel-size-raw.txt").read_text(encoding="utf-8"))
if not np.isfinite(pixel_size) or pixel_size <= 0:
    raise ValueError("Supply the actual H&E pixel size")
if superst_dir.exists():
    raise FileExistsError("Use a new output directory for SuperST preparation")


# Prepare a separate SuperST input directory

Start with the complete output from `pipeline.preprocessing.prepare_inputs`. This notebook preserves the input counts, locations and gene list, and pads the spatial images on their bottom and right edges. It retains the original H&E and HR reference. Pass the saved `target_s` to the SuperST launcher.


In [ ]:
he = tifffile.imread(prepared_dir / "he-raw.tiff")
plt.imshow(he)
print(he.shape)


In [ ]:
num_px_height = round(he.shape[0] * pixel_size / 8)
num_px_width = round(he.shape[1] * pixel_size / 8)
target_s = round(64*(max(num_px_height, num_px_width)//64+1))
blank_height = round(8 / pixel_size * (target_s - num_px_height))
blank_width = round(8 / pixel_size * (target_s - num_px_width))
print("target_s:\t" + str(target_s))
print("num_px_height:\t" + str(num_px_height))
print("num_px_width:\t" + str(num_px_width))
print("blank_height:\t" + str(blank_height))
print("blank_width:\t" + str(blank_width))

In [ ]:
width = he.shape[1]
height = blank_height + 1
img = np.zeros((height, width, 3), np.uint8)
img_rgb = img.copy()
img_rgb[:, :, :] = [255, 255, 255]
he_sq = np.concatenate((he,img_rgb))

width = blank_width
height = he.shape[0] + blank_height + 1
img = np.zeros((height, width, 3), np.uint8)
img_rgb = img.copy()
img_rgb[:, :, :] = [255, 255, 255]
he_sq = np.concatenate((he_sq,img_rgb), axis=1)

In [ ]:
he_sq.shape

In [ ]:
plt.imshow(he_sq)
plt.axis('off')

In [ ]:
shutil.copytree(prepared_dir, superst_dir)
data_dir = superst_dir / "spatial"
data_dir.mkdir(exist_ok=True)
tifffile.imwrite(data_dir / "tissue_hires_image.tiff", he_sq)
(superst_dir / "superst-target-size.txt").write_text(str(target_s), encoding="utf-8")


In [ ]:
he_sq_pil = Image.fromarray(he_sq)
he_sq_pil.save(data_dir / "tissue_hires_image.png", format="PNG")
he_sq_pil.save(data_dir / "tissue_lowres_image.png", format="PNG")
